In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(override=True)

In [ ]:
open_router_api_key = os.environ.get("OPEN_ROUTER_API_KEY")

if open_router_api_key is None:
    raise ValueError("OPENROUTER_API_KEY environment variable is not set")
else:
    print("OPENROUTER_API_KEY environment variable is set")

In [ ]:
llm = ChatOpenAI(
    model_name="openrouter/free",
    api_key=open_router_api_key,
    base_url="https://openrouter.ai/api/v1",
    temperature=0.5
)

In [ ]:
response = llm.invoke("Should we refund the user:123 5000 dollars?")

print(response.content)

approver = input("Approve? (y/n)")

if approver == "y":
    print("Approving refund")
else:
    print("Refund denied")

In [ ]:
from typing import TypedDict

class AgentState(TypedDict):
    question: str
    answer: str
    is_cricket: bool

In [ ]:
def answer_question(state: AgentState):
    question = state["question"]
    print(f"Question: {question}")
    
    return { "answer" : f"The answer to {question} is 42" }

In [ ]:
def analyze_question(state: AgentState):
    question = state["question"]

    if "cricket" in question.lower():
        return {"is_cricket": True}
    return {"is_cricket": False}

In [ ]:
def route_question(state: AgentState):
    is_cricket = state["is_cricket"]
    if is_cricket:
        return "answer"
    return "reject"

In [ ]:
def reject_question(state: AgentState):
    return {"answer": "I am only answer the cricket related questions"}

In [ ]:
from langgraph.graph import StateGraph


graph = StateGraph(AgentState)

graph.add_node("answer", answer_question)
graph.add_node("analyze", analyze_question)
graph.add_node("reject", reject_question)



In [ ]:
from langgraph.graph import START, END

graph.add_edge(START, "analyze")
graph.add_conditional_edges(
    "analyze",
    route_question,
    {
        "answer": "answer",
        "reject": "reject"
    }
)
graph.add_edge("answer", END)
graph.add_edge("reject", END)

app = graph.compile()

In [ ]:
result = app.invoke({
    "question": "Who is the king of cricket?",
    "answer": ""
})

In [ ]:
print(result)